# 03. Chạy lần lượt tất cả thí nghiệm trong một phiên

**Trước khi chạy:** Settings: Accelerator = GPU, Internet = On, Add Input = `seedlingstemrow`. Sau đó bấm **Save Version > Save & Run All (Commit)** để Kaggle chạy nền (giới hạn một phiên khoảng 12 giờ), bạn tắt trình duyệt vẫn chạy tiếp.

Thứ tự ưu tiên: E_main, E0 (baseline), E2, E1. Mỗi thí nghiệm xong là nén kết quả ngay, nên nếu hết giờ hoặc một thí nghiệm lỗi thì các thí nghiệm trước đó vẫn còn nguyên. Kết quả nằm ở tab *Output* của phiên bản đã lưu (mỗi thí nghiệm một file zip).

In [ ]:
# Bước 1: tải mã nguồn của bạn từ GitHub (sửa GITHUB_USER cho đúng, repo phải để chế độ Public)
import os, sys, subprocess
GITHUB_USER = "TEN_GITHUB_CUA_BAN"
TEN_REPO = "ssr-btl-thigiacmay"
DUONG_DAN = f"/kaggle/working/{TEN_REPO}"
if not os.path.isdir(DUONG_DAN):
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{GITHUB_USER}/{TEN_REPO}.git", DUONG_DAN], check=True)
else:
    subprocess.run(["git", "-C", DUONG_DAN, "pull"], check=True)   # lấy bản mới nhất sau mỗi lần bạn push
sys.path.insert(0, f"{DUONG_DAN}/src")
print("Mã nguồn sẵn sàng:", sorted(os.listdir(f"{DUONG_DAN}/src")))

In [ ]:
# Chuẩn bị một lần: vá repo, thu nhỏ dữ liệu (lần đầu lâu), ghi thông tin môi trường
import setup_env, ssr_lib as L
setup_env.cai_dat()
L.chuan_bi_du_lieu()
L.ghi_moi_truong()

In [ ]:
# Danh sách thí nghiệm theo thứ tự ưu tiên (có thể bỏ bớt hoặc thêm "E3")
import time
DANH_SACH = ["E_main", "E0", "E2", "E1"]
cau_hinh = L.tai_cau_hinh(f"{DUONG_DAN}/configs/experiments.json")
tong_ket = {}
t_bat_dau = time.time()
for ma in DANH_SACH:
    print("=" * 70)
    print(f"BẮT ĐẦU {ma}, đã trôi qua {round((time.time() - t_bat_dau) / 60)} phút")
    try:
        tong_ket[ma] = L.chay_thi_nghiem(ma, cau_hinh)
    except Exception as loi:                       # một thí nghiệm lỗi không làm mất các thí nghiệm khác
        print(f"LỖI khi chạy {ma}: {type(loi).__name__}: {loi}")
        tong_ket[ma] = False
    try:
        L.nen_ket_qua(ma)                          # nén ngay để không mất nếu phiên kết thúc sớm
    except Exception as loi:
        print(f"Không nén được {ma}: {loi}")
print("=" * 70)
print("TỔNG KẾT:", {k: ("thành công" if v else "LỖI") for k, v in tong_ket.items()})
print(f"Tổng thời gian: {round((time.time() - t_bat_dau) / 60)} phút")

In [ ]:
# Số liệu epoch cuối của từng thí nghiệm
for ma in DANH_SACH:
    L.tom_tat(ma)